# VLADE · Colab L4 24GB 학습

S0, S1 또는 legacy Plan 실험 중 한 작업만 선택합니다. GPU 적합성·메모리·속도는 파일럿에서 확인합니다.
HF_TOKEN Secret을 허용하고 plan → setup → preflight → pilot → train 순서로 실행합니다.
s2_legacy_plan은 과거 자연어 Plan 경로이며 현재 표현 증류 S2와 다릅니다.
legacy Plan은 preflight 다음 probe에서 실제 플랜을 확인합니다.
검증된 전체 액션 cache가 있으면 s1_cache를 선택합니다. 없으면 s1_online을 사용하거나
별도 cache_teacher 작업으로 생성·검증·HF 게시합니다. cache를 자동 생성하거나 대체하지 않습니다.
온라인 Teacher는 Student와 번갈아 GPU에 적재하므로 충분한 CPU RAM과 전송 시간이 필요합니다.
기존 A100 실행을 재개할 때는 원래 소스·설정을 유지하고 이 프로필로 바꾸지 마십시오.
세부 제한은 [L4 안내](../docs/environment/colab_l4_training.md)를 따릅니다.


In [ ]:
from pathlib import Path
import os, re, subprocess, sys, time
os.environ.setdefault('VLADE_SESSION_STARTED_UNIX', str(time.time()))
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
sys.dont_write_bytecode = True
SOURCE_REF = 'refs/heads/main'  # 재개는 최초 실행의 SOURCE_COMMIT을 지정합니다.
REPOSITORY = 'https://github.com/eastha10/VLADE.git'
SOURCE_COMMIT = SOURCE_REF if re.fullmatch(r'[0-9a-f]{40}', SOURCE_REF) else subprocess.check_output(['git','ls-remote',REPOSITORY,SOURCE_REF], text=True).split()[0]
if not re.fullmatch(r'[0-9a-f]{40}', SOURCE_COMMIT):
    raise ValueError('불변 Git commit이 필요합니다.')
CODE = Path('/content/vlade_sources') / SOURCE_COMMIT
if any((name == 'vlade' or name.startswith('vlade.')) and getattr(module,'__file__',None) and not Path(module.__file__).resolve().is_relative_to(CODE) for name,module in list(sys.modules.items())):
    raise RuntimeError('다른 소스가 로드되어 있습니다. Colab 런타임을 다시 시작하십시오.')
if not CODE.exists():
    subprocess.run(['git','clone','--no-checkout','--filter=blob:none',REPOSITORY,str(CODE)],check=True)
    subprocess.run(['git','-C',str(CODE),'fetch','--depth','1','origin',SOURCE_COMMIT],check=True)
    subprocess.run(['git','-C',str(CODE),'checkout','--detach',SOURCE_COMMIT],check=True)
sys.path[:0] = [str(CODE/'src'),str(CODE)]
from vlade.utils.source_identity import source_identity
assert source_identity(CODE,require_clean=True)['git_commit']==SOURCE_COMMIT
print('SOURCE_COMMIT:',SOURCE_COMMIT)


In [ ]:
from vlade.research.distillation import new_run_group

PROFILE = 's0'  #@param ['s0', 's1_cache', 's1_online', 's2_legacy_plan']
CONFIGS = {
    's0': 'configs/experiments/qwen_vla_s0_l4.json',
    's1_cache': 'configs/experiments/qwen_vla_action_kd_l4_cache.json',
    's1_online': 'configs/experiments/qwen_vla_action_kd_l4.json',
    's2_legacy_plan': 'src/vlade/legacy/plan_cot/configs/experiments/qwen_vla_plan_kd_l4.json',
}
CONFIG = CONFIGS[PROFILE]
WORKSPACE = '/content/vlade_l4'
RUN_GROUP = globals().get('RUN_GROUP') or new_run_group('l4-' + PROFILE)
ACTION = 'plan'  #@param ['plan', 'setup', 'preflight', 'probe', 'pilot', 'train', 'retry']
OVERRIDES = {}
RESUME_REVISIONS = {}  # 예: {'S1': '40자리 HF commit SHA'}


In [ ]:
from vlade.research.distillation import job

job(CODE / CONFIG, workspace=WORKSPACE, run_group=RUN_GROUP, action=ACTION,
    overrides=OVERRIDES, resume_revisions=RESUME_REVISIONS, colab=True)
